# ComicBookGenerator ? Colab GPU

Run each cell in order. This notebook always clones the latest GitHub code, fixes the Colab Pillow compatibility issue before importing the backend, checks the GPU, starts FastAPI with a readiness loop, and saves outputs under `/content/ComicBookGenerator/outputs`.

In [ ]:
%cd /content
!rm -rf ComicBookGenerator
!git clone https://github.com/CallmeTruong/Comic_Studio.git ComicBookGenerator
%cd /content/ComicBookGenerator
!git rev-parse --short HEAD

In [ ]:
# Keep Colab's CUDA PyTorch; install the Python packages used by the project.
!pip install -q --upgrade --force-reinstall "Pillow>=11.3.0"
!pip install -q fastapi uvicorn python-dotenv numpy diffusers transformers accelerate safetensors sentencepiece peft compel huggingface-hub langgraph langchain-core langchain-openai openai rembg fonttools onnxruntime-gpu
import PIL
print("Pillow:", PIL.__version__)
print("If this is the first run after a Pillow upgrade, use Runtime ? Restart session, then run all cells again.")

In [ ]:
!nvidia-smi
import torch
print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
if not torch.cuda.is_available(): raise RuntimeError("Enable Runtime ? Change runtime type ? GPU in Colab.")
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
from pathlib import Path
drive_models=Path("/content/drive/MyDrive/ComicBookGenerator/models")
print("Drive model directory:", drive_models)
print("Put comicBabes_v2.safetensors under Drive/.../models/base for SD1.5.")

In [ ]:
# Optional API credentials. Store them in Colab Secrets, not in this notebook.
import os
try:
  from google.colab import userdata
  key=userdata.get("OPENAI_API_KEY")
  if key: os.environ["OPENAI_API_KEY"]=key
except Exception: pass
os.environ.setdefault("OPENAI_BASE_URL", "https://api.openai.com/v1")
MODEL="sdxl_dreamshaper" # change to "sd15" when your checkpoint is in Drive
os.environ["PYTHONPATH"]="/content/ComicBookGenerator"
from core.model_registry import ensure_model
profile,model_path=ensure_model(MODEL)
print(profile["label"], model_path)

## Start the backend

This cell does not assume a fixed 25-second startup. It waits until `/api/capabilities` responds, or prints the backend log when startup fails.

In [ ]:
import subprocess,time,requests,sys,os
log_path="/content/ComicBookGenerator/colab_backend.log"
log_file=open(log_path,"w",encoding="utf-8")
api_process=subprocess.Popen([sys.executable,"api.py"],cwd="/content/ComicBookGenerator",stdout=log_file,stderr=subprocess.STDOUT,text=True,env=os.environ.copy())
url="http://127.0.0.1:8000/api/capabilities"
deadline=time.time()+180
last_error=None
while time.time()<deadline:
  if api_process.poll() is not None:
    log_file.flush(); print(open(log_path,encoding="utf-8").read())
    raise RuntimeError(f"Backend exited with code {api_process.returncode}")
  try:
    r=requests.get(url,timeout=5); r.raise_for_status(); print("Backend ready:",r.json()); break
  except requests.RequestException as exc:
    last_error=exc; time.sleep(3)
else:
  log_file.flush(); print(open(log_path,encoding="utf-8").read())
  raise TimeoutError(f"Backend did not become ready: {last_error}")

In [ ]:
# Integration test. Use 20 steps first; raise STEPS after the API works.
STEPS=20
payload={"prompt":"Write a short four-panel comic in natural English about one careful robot delivering one parcel. Use one setting, clear cause and effect, short dialogue, and a visual punchline.","model":MODEL,"steps":STEPS,"guidance":7.0,"lora":"","seed":"12345","pageCount":1}
events=[]
with requests.post("http://127.0.0.1:8000/api/generate",json=payload,stream=True,timeout=1800) as response:
  response.raise_for_status()
  for line in response.iter_lines(decode_unicode=True):
    if line and line.startswith("data: "):
      event=line[6:]; events.append(event); print(event)
Path("outputs/colab_events.json").write_text(json.dumps(events,ensure_ascii=False,indent=2),encoding="utf-8")

In [ ]:
from pathlib import Path
pages=sorted(Path("outputs").glob("comic_page_*.png"),key=lambda p:p.stat().st_mtime,reverse=True)
print("Generated pages:")
print(*[str(p.resolve()) for p in pages[:5]],sep="\n")
print("Backend log:",log_path)